In [1]:
from semanticscholar import SemanticScholar
from habanero import Crossref
import pandas as pd

In [2]:
# TODO: Update search keywords
primary_keywords = " | ".join(
    ["Generative AI", "LLM", "GenAI", "large language model", "codex", "gpt-3", "gpt-4"]
)  # and so on...
secondary_keywords = " | ".join(
    [
        "overreliance",
        "misinformation",
        "accessbility",
        "privacy",
        "enviromental",
        "explainability",
        "trustworthy",
        "responsible",
    ]
)  # and so on...
others = ["mitigation"]

year = "2019-"

all_keywords = f"({primary_keywords}) + ({secondary_keywords})"

In [3]:
# Prepare DataFrame to store the results
columns = [
    'PaperTitle',
    'DOI',
    'Authors',
    'Abstract',
    'Publisher',
    'SemanticScholarUrl',
    'DoiUrl',
    'PublicationDate',
    'FieldOfStudy',
    'Conference-Journal',
    'PublicationTypes',
    'SearchString',
    'CitationCount',
    'SearchedFrom'
]

In [4]:
crossref = Crossref()

In [8]:
def extract_data(sch_results, results: list, query: str):
    for sch_paper in sch_results:
        try:
            crossref_paper = crossref.works(ids=sch_paper["externalIds"].get("DOI"))
        except Exception as e:
            crossref_paper = None

        title = sch_paper["title"]
        doi = sch_paper["externalIds"].get("DOI")

        authors = None
        # author name and affiliation
        if crossref_paper is not None:
            authors = crossref_paper.get("message").get("author")
        if authors is not None:
            for i in range(len(authors)):
                author = authors[i]
                author_name = author.get("given", "") + " " + author.get("family", "")
                affiliation = author.get("affiliation", "No Affiliation")
                affiliations = author.get("affiliation", [])
                school_names = (
                    [affil.get("name") for affil in affiliations]
                    if affiliations
                    else ["No Affiliation"]
                )
                # Create a new dictionary with only 'name' and 'affiliation'
                authors[i] = {
                    "name": author_name.strip(),
                    "affiliation": school_names,
                }
        else:
            authors = sch_paper["authors"]
            for i in range(len(sch_paper["authors"])):
                author = sch_paper["authors"][i]
                sch_paper["authors"][i] = {
                    "name": author.get("name", "No Name"),
                    "affiliation": author.get("affiliation", "No Affiliation"),
                }

        abstract = sch_paper["abstract"]
        sch_url = sch_paper["url"]
        doi_url = f"https://doi.org/{doi}"
        publication_date = sch_paper["publicationDate"]
        fields_of_study = sch_paper["fieldsOfStudy"]
        venue = sch_paper["venue"]

        # publisher
        if crossref_paper is not None:
            publisher = crossref_paper.get("message").get("publisher")
        elif doi and "arxiv" in doi.lower():
            publisher = "arXiv"
        else:
            publisher = None

        # paper type
        if crossref_paper is not None:
            paper_type = [crossref_paper.get("message").get("type")]
        else:
            paper_type = sch_paper["publicationTypes"]

        citation_count = sch_paper["citationCount"]
        # TODO: paper keywords missing
        # TODO: paper type is conference/journal for arxiv papers
        # TODO: conference-journal name mismatch with publisher, i.e., for paper with name"ChatGPT in education: A discourse analysis of worries and concerns on social media", the conference name is "International Conference on Artificial Intelligence in Education", but the publisher is "Arxiv" (becauseit queryed from arxiv), need "Springer" instead.

        new_paper = {
            "PaperTitle": title,
            "DOI": doi,
            "Authors": authors,
            "Abstract": abstract,
            "Publisher": publisher,
            "SemanticScholarUrl": sch_url,
            "DoiUrl": doi_url,
            "PublicationDate": publication_date,
            "FieldOfStudy": fields_of_study,
            "Conference-Journal": venue,
            "PublicationTypes": paper_type,
            "SearchString": query,
            "CitationCount": citation_count,
            "SearchedFrom": "Semantic Scholar",
        }
        print(new_paper)
        results.append(new_paper)

In [9]:
import requests

results = []
query = all_keywords
fields = [
    "title",
    "externalIds",
    "authors",
    "abstract",
    "url",
    "publicationDate",
    "fieldsOfStudy",
    "venue",
    "publicationTypes",
    "citationCount",
]

# send http request to api
api_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
headers = {"Content-Type": "application/json"}
params = {"query": query, "year": year, "fields": ",".join(fields)}
response = requests.get(api_url, headers=headers, params=params)

# parse response
response_json = response.json()
sch_results = response_json["data"]
extract_data(sch_results, results, query)
print(len(sch_results))

# TODO: Debug why search results are hanging (15 mins for 1000 results) -- maybe the token is not working? The API said if there is extra results, a token will be present to go to the next page
# while response_json["token"] is not None or response_json["token"] != "":
#     params["token"] = response_json["token"]
#     response = requests.get(api_url, headers=headers, params=params)
#     response_json = response.json()
#     sch_results = response_json["data"]
#     print(len(sch_results))
#     extract_data(sch_results, results, query)

results_df = pd.DataFrame(results, columns=columns)
results_df.to_csv("data/initial-scrape-result.csv", index=False)

{'PaperTitle': 'More Than Just Facts: Promoting Civic Media Literacy in the Era of Outrage', 'DOI': '10.1080/0161956X.2019.1553582', 'Authors': [{'name': 'Ellen Middaugh', 'affiliation': ['San José State University, San Jose, California, USA']}], 'Abstract': 'Abstract Amid rising concerns about “fake news,” efforts have emerged to explain the spread and impact of misinformation on youth civic engagement. These efforts have focused primarily on the role of social media in exposing youth to factually inaccurate civic information and the factors that influence the ability to discern the accuracy of such information. A less explored aspect has been the impact of the rise of “outrage language,” defined as language that evokes strong emotional responses (e.g., fear, anger, disgust) that communications scholars have documented as playing a larger role in political discourse over the past few decades (Berry & Sobieraj, 2014). This article draws on three recent studies of digital media and yout

KeyboardInterrupt: 

In [13]:
# semantic scholar test
sch = SemanticScholar()
sch_results = sch.get_paper("10.1177/030631284014003004")


In [12]:
#corssref test

cr = Crossref()
cr_results = cr.works(query = "Education and Information Technologies : Official Journal of the IFIP technical committee on Education")